# App Store Reviews Scraper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/app-store-reviews.ipynb)

**What:** Apple App Store reviews at $0.50 per 1,000, verified by 85 passing tests. Review text, stars, app version, date, votes, plus rating, rating count, price, genre and release notes.

**Why:** run the [`app-store-reviews`](https://apify.com/transparent_meteorite/app-store-reviews) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Review author names are dropped from the table; only ratings and review-level fields are shown.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/app-store-reviews) for every option.

In [2]:
run_input = {   'appIds': ['https://apps.apple.com/us/app/spotify-music-and-podcasts/id324684580'],
    'countries': ['us'],
    'maxReviewsPerApp': 5,
    'includeAppDetails': True}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/app-store-reviews').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> Status: RUNNING, Message: 


[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> 2026-10-13T22:17:00.833Z ACTOR: Pulling container image of build dMlVDGVfygLAmi1sI from registry.


[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> 2026-10-13T22:17:00.836Z ACTOR: Creating container.


[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> 2026-10-13T22:17:00.903Z ACTOR: Starting container.


[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> 2026-10-13T22:17:00.904Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> 2026-10-13T22:17:02.657Z Done: {"apps":1,"appCountryPairs":1,"reviews":5,"appRows":1,"charged":{"review-returned":5,"app-details":1},"skippedInvalid":0,"notFound":[],"perApp":1,"chargeLimitReached":false,"warnings":[]}


[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> 2026-10-13T22:17:02.657Z INFO  [Status message]: Done: 6 items


[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> 2026-10-13T22:17:02.721Z Thanks for using App Store Reviews Scraper. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/8y9xfYCc1Vw62BPh4/issues) with the run URL and we will reply there.


[apify.app-store-reviews runId:WvvSypCm5c2aRz5DP] -> Status: SUCCEEDED, Message: Done: 6 items


Run status: SUCCEEDED | items: 6


## 4. Tabulate

Review author names are dropped from the table; only ratings and review-level fields are shown.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["type", "appName", "title", "rating", "version", "date", "developer", "genre", "ratingAverage"] if c in df.columns]
df[SHOW]

,type,appName,title,rating,version,date,developer,genre,ratingAverage
0,app,Spotify: Music and Podcasts,NaN,NaN,9.1.90,NaN,Spotify,Music,4.77
1,review,Spotify: Music and Podcasts,fagit,1.0,9.1.90,2026-10-12T19:29:14.000Z,NaN,NaN,NaN
2,review,Spotify: Music and Podcasts,It’s Okay,3.0,9.1.90,2026-10-12T19:25:59.000Z,NaN,NaN,NaN
3,review,Spotify: Music and Podcasts,I’m so sorry about this but..,3.0,9.1.90,2026-10-12T19:25:18.000Z,NaN,NaN,NaN
4,review,Spotify: Music and Podcasts,So lit,5.0,9.1.90,2026-10-12T19:25:10.000Z,NaN,NaN,NaN
5,review,Spotify: Music and Podcasts,Spotify,5.0,9.1.90,2026-10-12T19:24:43.000Z,NaN,NaN,NaN


## 5. A quick analysis

In [5]:
rev=df[df['type']=='review'].copy()
rev['rating']=pd.to_numeric(rev['rating'],errors='coerce')
print(rev['rating'].value_counts().sort_index())
print('Mean rating of sampled reviews:',round(rev['rating'].mean(),2))

rating
1.0    1
3.0    2
5.0    2
Name: count, dtype: int64
Mean rating of sampled reviews: 3.4


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [app-store-reviews on the Apify Store](https://apify.com/transparent_meteorite/app-store-reviews).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.